# Problem Set 2: Who Hosts the World's Refugees?

QSS 20 · Modern Statistical Computing · Fall 2026

**Out:** Thursday, October 1 · **Due:** Thursday, October 8, 11:59pm

## Overview

If you followed the news in 2015, when asylum applications to Europe surged,
or in 2022, when millions fled Ukraine, you might think that rich countries
host most of the world's refugees. The United Nations refugee agency (UNHCR)
keeps a register that can check that impression. It records how many refugees
from each origin country live in each host country, every year. The answer
depends on the denominator: total refugees, or refugees per resident. You will
compute both.

The set draws on lectures 4 through 7. You will select rows, group and
aggregate, reshape a wide table, and merge two sources on a two-column key.
You will also handle missing values and draw figures that show the
denominator argument.

**Submission.** Copy `ps2_blank.ipynb` as `ps2_mywork.ipynb` and work in it.
Keep `{{{< pagebreak >}}}` so page tagging on Gradescope stays simple.
Once you're finished, click "Kernel", "Restart Kernel and Clear Outputs
of All Cells", and then "Restart Kernel and Run All Cells", then
export your notebook to PDF with
`quarto render ps2_mywork.ipynb --to typst`.
Upload `ps2_mywork.ipynb` and `ps2_mywork.pdf` to Gradescope.

**Data.** Two sources sit in `data/raw/`. The refugee table has one row per
**origin country, host country, and year**:

| Field | Meaning |
|---|---|
| `year` | 2000 through 2024 |
| `origin`, `origin_iso` | the country refugees fled, as a name and a three-letter ISO code |
| `asylum`, `asylum_iso` | the host country, as a name and a three-letter ISO code |
| `refugees` | people recognized as refugees under UNHCR's mandate and present in the host country at the end of the year |
| `asylum_seekers` | people whose asylum claim is still pending (not used in this set) |

The population table, in `data/raw/wb_population/`, is the World Bank's total
population by country, with **one row per country and one column per year**.
See the Appendix for where both files come from and for one boundary of the
refugee count.

In [ ]:
import glob
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
plt.rcParams['figure.dpi'] = 200
plt.rcParams['savefig.dpi'] = 300

REF ="data/raw/unhcr_refugees_2000_2024.csv"
WB = glob.glob("data/raw/wb_population/API_SP.POP.TOTL*.csv")[0]

{{< pagebreak >}}

## 1. Load and check the two files (14 points)

**1.1 (6 points).** Load the refugee file as `refugees`. Report its shape,
each column's dtype, and the first and last year. State the row unit in one
sentence. Which two fields, together with `year`, should identify a row
uniquely? Verify with `.duplicated(subset=[...]).sum()`.

{{< pagebreak >}}

**1.2 (8 points).** In the real world, not all CSV files are stored properly. Here you're seeing one example. A plain `pd.read_csv(WB)` stops with a `ParserError`.
Print the file's first 10 raw lines instead and check what's causing the `ParserError`:

```python
with open(WB, encoding="utf-8-sig") as f:
    for line in list(f)[:10]:
        print(repr(line[:80]))
```

What do the first four lines hold? Load the file as `population_wide` with
the `skiprows` argument of `pd.read_csv`, and report its shape. Then use
`.isin` to print the `Country Name` and `2024` value for the codes `WLD`,
`EUU`, and `HIC`. What does one row represent for these three?

{{< pagebreak >}}

## 2. Select rows (12 points)

In each task, build the selection, then **check it**: print the values or
counts that show the filter kept the right rows.

**2.1 (6 points).** Select the rows for **2024** with **at least one**
refugee. Report how many rows remain and how many distinct host countries
they cover. Check the selection by printing its years and its smallest
`refugees` value. How many 2024 rows did the second condition remove, and why
could a row with zero refugees exist at all? (Look at the field table.)

{{< pagebreak >}}

**2.2 (6 points).** The war in Syria began in 2011. Select the rows for
refugees **from Syria** (`origin_iso` equal to `"SYR"`) in the years **2011
through 2024**, hosted in **Türkiye, Lebanon, Jordan, or Germany** (`TUR`,
`LBN`, `JOR`, `DEU`). Use `.between` for the years and `.isin` for the hosts,
in one combined mask. Check it by printing the selected years and hosts. Then sort the
selection by `refugees` and use `.iloc` to print its three largest rows. In
which year and host was the count largest?

{{< pagebreak >}}

## 3. Group by host and year (18 points)

The refugee table's row unit is finer than the question. "Who hosts
refugees" needs one row per **host country per year**, summed over origins.

**3.1 (8 points).** Build that table as `hosts`: `groupby` on `asylum`,
`asylum_iso`, and `year` (with `as_index=False`), and named aggregation with
two columns. `refugees` is the sum of refugees; `origins` is the number of
origin countries with at least one refugee, from your own function or
`lambda`. Check the result: print the number of rows before and after, and whether
total refugees match.
Then compute the world total for each year, and report the totals for 2011
and 2024 and the factor by which it grew.

{{< pagebreak >}}

**3.2 (10 points).** Add a column `share` to `hosts`: each host's refugees as
a share of the **world total in the same year**, computed with `transform`.
Report the eight largest hosts of 2024 with their `refugees` and `share`, and
the share of the world's refugees those eight held together.

Then use `pivot_table` to build a table of `refugees` with one row per
`asylum_iso` and one column per year. How many cells are `NaN`, and what does
a `NaN` mean here? Should it become 0? Decide, rebuild the table with that
choice, and defend it in two sentences. (Count the `hosts` rows whose
`refugees` is 0, and compare them with a `NaN` cell.)

{{< pagebreak >}}

## 4. Reshape and merge (20 points)

**4.1 (8 points).** The population table stores years as **columns**; the
merge needs them as **rows**. Use `melt` to make a long table `population`
with columns `country`, `iso3`, `year`, and `population` for 2000 through
2024. The years arrive as text, so convert `year` with `.astype(int)`. Check
the reshape: print the expected number of rows (rows times years), the actual
number, and the number of missing `population` values. Which row are the missing values from?

{{< pagebreak >}}

**4.2 (6 points).** Write the merge plan before you run it: the row unit of
`hosts` and of `population`, the **two-column key**, the relationship you
expect (one-to-one, many-to-one, ...), and which table's rows must all
survive. Then merge `population` onto `hosts` with `how="left"`,
`validate="one_to_one"`, and `indicator=True`, as `merged`. Report the counts
in `_merge`. Which host entities never match, how many rows does each have,
and how many refugees do those rows hold?

{{< pagebreak >}}

**4.3 (6 points).** Run the same merge with `how="inner"`, `how="right"`,
and `how="outer"`, and print the number of rows each returns next to the
left join's count. Explain each count from the numbers in 4.1 and 4.2. What
kinds of rows does the outer join add that the left join leaves out? Finally,
add a column `per_1000` to `merged`: refugees per 1,000 residents.

{{< pagebreak >}}

## 5. Two figures, two denominators (16 points)

**5.1 (6 points).** Draw the world refugee total from 2000 through 2024 as a
line chart with `fig, ax`. Label both axes, show the total in millions, and
add one `ax.annotate` that marks an event visible in the line. Describe the
line's shape in one or two sentences.

{{< pagebreak >}}

**5.2 (10 points).** Now make the denominator argument for 2024. Keep only
hosts with at least **one million** residents, and say in one sentence why a
per-capita comparison needs a population floor.

1. Draw a histogram of `per_1000` across these hosts at two bin widths, as
   two panels. Describe the shape, and print the mean and the median. Which
   one describes a typical host better, and why?
2. Draw one figure with two `barh` panels: the ten largest hosts by
   **total refugees**, and the ten largest by **refugees per 1,000
   residents**. Label both panels.

Who hosts the world's refugees under each measure? Answer in two or three
sentences.

{{< pagebreak >}}

## 6. Missing values and limits (10 points)

**6.1 (10 points).** First, with code: how many rows of `merged` have no
`per_1000`, why is it missing for exactly those rows, and how many refugees
do they hold? Did your Section 5 figures keep or drop them, and does that
change any conclusion? Then write three sentences, each naming one distinct
limit of these counts as a measure of "who hosts the world's refugees." The
Appendix names one boundary; the field table supplies others.

{{< pagebreak >}}

## Problem Set Reflections (10 points)

Every problem set ends with these reflections. Answer them honestly. You are
graded on the specificity and honesty in the reflections you show, not on how
much AI you used or did not use. You cannot use AI for this part, and we are
not checking for spelling or grammar. Write as much as each answer needs.

Write your answer under each heading.

### AI use

List each substantive use of an AI assistant: the tool and model, what you
asked for, what it gave back, how you checked it, and whether you kept or
rejected the result. Write "None" if you used none.

*Your answer:*

### People consulted

Who you talked to about this assignment, and what about.

*Your answer:*

### Where you got stuck

The step that blocked you, and what got you unstuck.

*Your answer:*

### What you understand now

One thing you understand now that you did not when you started.

*Your answer:*

### Hours spent

An honest estimate of the time this took.

*Your answer:*

{{< pagebreak >}}

## Appendix: where the data came from

The refugee table comes from the UNHCR Refugee Data Finder
(`api.unhcr.org/population/v1/population/`, retrieved 2026-08-27 UTC). It
covers 2000 through 2024, every origin-host pair, and carries the `refugees`
and `asylum_seekers` variables.

The population table is the World Bank's bulk download of indicator
`SP.POP.TOTL`, left exactly as the World Bank publishes it, which is why its
header does not start on line 1.

One boundary you need for Section 6: UNHCR's refugee statistics cover people
under UNHCR's mandate. Palestinian refugees registered with UNRWA, a separate
UN agency, are not in this register.